# Slide Exercise 01: Expanded TF-IDF Movie Recommender

This is the refined version of `TFIDF_MovieRecommender_Expanded.ipynb`.

Learning objectives:
- Build a stronger TF-IDF content representation from title, genres, director, description, and keywords.
- Generate Top-N similar movies.
- Explain recommendations with shared weighted terms.

Main functions used:
- `TfidfVectorizer(...)`: converts text into weighted term vectors.
- `fit_transform(...)`: learns the vocabulary and creates the movie-term matrix.
- `cosine_similarity(...)`: compares movie vectors by angle.
- `argsort()`: sorts similarity scores to create a ranking.


Load the shared Chapter 2 dataset. We keep the dataset small so students can inspect every intermediate result.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


,movie_id,title,genres,director,year,duration_min,rating,family_friendly,description,keywords
0,1,Inception,Sci-Fi|Thriller|Action,Christopher Nolan,2010,148,8.8,0,A thief enters layered dreams to plant an idea...,dreams heist subconscious mind-bending
1,2,Interstellar,Sci-Fi|Adventure|Drama,Christopher Nolan,2014,169,8.7,0,Astronauts travel through a wormhole to find a...,space exploration wormhole survival family
2,3,Titanic,Romance|Drama,James Cameron,1997,195,7.9,0,A young couple from different social classes f...,romance ship tragedy historical
3,4,The Matrix,Sci-Fi|Action,The Wachowskis,1999,136,8.7,0,A hacker discovers that reality is a simulated...,simulation hacker reality action cyberpunk
4,5,Toy Story,Animation|Adventure|Comedy|Family,John Lasseter,1995,81,8.3,1,A cowboy doll feels threatened when a space ra...,toys friendship family adventure


Create one clean text field per movie. In real systems this step is often called metadata fusion.


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

movies["content"] = (
    movies["title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["director"] + " " +
    movies["description"] + " " +
    movies["keywords"]
).apply(clean_text)

movies[["title", "content"]].head()


,title,content
0,Inception,inception sci-fi thriller action christopher n...
1,Interstellar,interstellar sci-fi adventure drama christophe...
2,Titanic,titanic romance drama james cameron a young co...
3,The Matrix,the matrix sci-fi action the wachowskis a hack...
4,Toy Story,toy story animation adventure comedy family jo...


Fit TF-IDF and compute a movie-by-movie similarity matrix.


In [3]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
tfidf_matrix = vectorizer.fit_transform(movies["content"])
similarity = cosine_similarity(tfidf_matrix)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
pd.DataFrame(similarity, index=movies["title"], columns=movies["title"]).round(2)


TF-IDF matrix shape: (12, 348)


title,Inception,Interstellar,Titanic,The Matrix,Toy Story,Finding Nemo,The Dark Knight,The Martian,The Notebook,Paddington,Gravity,La La Land
title,,,,,,,,,,,,
Inception,1.00,0.09,0.00,0.06,0.00,0.00,0.07,0.03,0.00,0.00,0.08,0.04
Interstellar,0.09,1.00,0.01,0.04,0.08,0.04,0.09,0.10,0.02,0.08,0.11,0.01
Titanic,0.00,0.01,1.00,0.00,0.00,0.04,0.01,0.00,0.18,0.02,0.01,0.14
The Matrix,0.06,0.04,0.00,1.00,0.00,0.00,0.03,0.03,0.00,0.00,0.03,0.00
Toy Story,0.00,0.08,0.00,0.00,1.00,0.10,0.00,0.06,0.02,0.17,0.01,0.00
Finding Nemo,0.00,0.04,0.04,0.00,0.10,1.00,0.00,0.05,0.00,0.09,0.00,0.00
The Dark Knight,0.07,0.09,0.01,0.03,0.00,0.00,1.00,0.00,0.02,0.00,0.01,0.01
The Martian,0.03,0.10,0.00,0.03,0.06,0.05,0.00,1.00,0.00,0.05,0.13,0.00
The Notebook,0.00,0.02,0.18,0.00,0.02,0.00,0.02,0.00,1.00,0.00,0.02,0.14


Define small reusable functions. The recommendation function ranks movies; the explanation function shows shared TF-IDF features.


In [4]:
def shared_weighted_terms(seed_idx, other_idx, top_n=6):
    terms = np.array(vectorizer.get_feature_names_out())
    seed_weights = tfidf_matrix[seed_idx].toarray().ravel()
    other_weights = tfidf_matrix[other_idx].toarray().ravel()
    shared_weight = np.minimum(seed_weights, other_weights)
    best = shared_weight.argsort()[::-1][:top_n]
    return ", ".join(terms[i] for i in best if shared_weight[i] > 0)

def recommend_similar_movies(title, n=5):
    seed_idx = movies.index[movies["title"].eq(title)][0]
    ranked = similarity[seed_idx].argsort()[::-1]
    rows = []
    for other_idx in ranked:
        if other_idx == seed_idx:
            continue
        rows.append({
            "input_movie": title,
            "recommended_movie": movies.loc[other_idx, "title"],
            "similarity": round(float(similarity[seed_idx, other_idx]), 3),
            "shared_terms": shared_weighted_terms(seed_idx, other_idx),
        })
        if len(rows) == n:
            break
    return pd.DataFrame(rows)

recommend_similar_movies("Interstellar")


,input_movie,recommended_movie,similarity,shared_terms
0,Interstellar,Gravity,0.108,"astronauts, survival, space, fi, sci fi, sci"
1,Interstellar,The Martian,0.097,"fi adventure, survival, space, fi, adventure, ..."
2,Interstellar,Inception,0.088,"nolan, christopher, christopher nolan, fi, sci..."
3,Interstellar,The Dark Knight,0.087,"drama christopher, nolan, christopher, christo..."
4,Interstellar,Toy Story,0.084,"new, family, adventure, space"


Interpretation:

Movies with shared terms such as `space`, `astronaut`, `sci-fi`, or a shared director move upward in the ranking. TF-IDF is still lexical, so it works best when related movies use overlapping vocabulary.

Student task:
1. Change the input movie to `Toy Story`.
2. Remove bigrams by setting `ngram_range=(1, 1)`. Did the ranking change?
